In [1]:
import GridMLM_tokenizers
from GridMLM_tokenizers import CSGridMLMTokenizer
from data_utils import CSGridMLMDataset, CSGridMLM_collate_fn
from torch.utils.data import DataLoader
from models import SEFiLMModel, ContrastiveSpaceModel, contrastive_normalized_loss
import torch
from torch.nn import CrossEntropyLoss
import os
import pickle
from data_utils import ContrastiveCollator
from train_utils import make_mixed_batch, full_to_partial_masking
from evaluation_utils import evaluate_contrastive_convergence
from tqdm import tqdm
from pprint import pprint

/home/maximos/.local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
source_name = 'lstm'
shared_dim = 64
batch_size = 16
device_name = 'cuda:0'

In [3]:
source_key = source_name + '_embeddings'

train_path = 'data/contrastive_dataset/CA_train.pickle'
val_path = 'data/contrastive_dataset/CA_test.pickle'

with open(train_path, 'rb') as f:
    train_dataset = pickle.load(f)
with open(val_path, 'rb') as f:
    val_dataset = pickle.load(f)

source_dim = train_dataset[0][source_key].shape[0]
transformer_dim = train_dataset[0]['transformer_embeddings'].shape[0]

print(source_name, ' - source_dim: ', source_dim)
print('transformer', ' - transformer_dim: ', transformer_dim)
print('shared', ' - shared_dim: ', shared_dim)

collator = ContrastiveCollator(pad_id=0)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collator)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=collator)

if device_name == 'cpu':
    device = torch.device('cpu')
else:
    if torch.cuda.is_available():
        device = torch.device(device_name)
    else:
        print('Selected device not available: ' + device_name)
# end device selection

contrastive_model = ContrastiveSpaceModel(source_dim, transformer_dim, shared_dim=shared_dim)
checkpoint = torch.load(f'saved_models/contrastive/{source_name}.pt', map_location=device_name)
contrastive_model.load_state_dict(checkpoint)
contrastive_model.to(device)

# contrastive_loss_fn = contrastive_normalized_loss
contrastive_loss_fn = torch.nn.MSELoss()

tokenizer = CSGridMLMTokenizer(
    fixed_length=80,
    quantization='4th',
    intertwine_bar_info=True,
    trim_start=False,
    use_pc_roll=True,
    use_full_range_melody=False
)

mask_token_id = tokenizer.mask_token_id
bar_token_id = tokenizer.bar_token_id

logits_loss_fn =CrossEntropyLoss(ignore_index=-100)

transformer_film_model = SEFiLMModel(
    chord_vocab_size=len(tokenizer.vocab),
    d_model=512,
    nhead=8,
    num_layers=8,
    grid_length=80,
    pianoroll_dim=tokenizer.pianoroll_dim,
    guidance_dim=shared_dim,
    device=device,
)
checkpoint = torch.load(f'saved_models/film/{source_name}.pt', map_location=device_name)
transformer_film_model.load_state_dict(checkpoint)
transformer_film_model.to(device)

transformer_lacta_model = SEFiLMModel(
    chord_vocab_size=len(tokenizer.vocab),
    d_model=512,
    nhead=8,
    num_layers=8,
    grid_length=80,
    pianoroll_dim=tokenizer.pianoroll_dim,
    guidance_dim=shared_dim,
    device=device,
)
checkpoint = torch.load(f'saved_models/lacta/{source_name}.pt', map_location=device_name)
transformer_lacta_model.load_state_dict(checkpoint)
transformer_lacta_model.to(device)

transformer_lacta_full_model = SEFiLMModel(
    chord_vocab_size=len(tokenizer.vocab),
    d_model=512,
    nhead=8,
    num_layers=8,
    grid_length=80,
    pianoroll_dim=tokenizer.pianoroll_dim,
    guidance_dim=shared_dim,
    device=device,
)
checkpoint = torch.load(f'saved_models/lacta_full/{source_name}.pt', map_location=device_name)
transformer_lacta_full_model.load_state_dict(checkpoint)
transformer_lacta_full_model.to(device)

lstm  - source_dim:  128
transformer  - transformer_dim:  512
shared  - shared_dim:  64


SEFiLMModel(
  (melody_proj): Linear(in_features=13, out_features=512, bias=True)
  (harmony_embedding): Embedding(355, 512)
  (dropout): Dropout(p=0.3, inplace=False)
  (encoder_layers): ModuleList(
    (0-7): 8 x TransformerEncoderLayerWithFiLM(
      (layer): TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
        )
        (linear1): Linear(in_features=512, out_features=2048, bias=True)
        (dropout): Dropout(p=0.3, inplace=False)
        (linear2): Linear(in_features=2048, out_features=512, bias=True)
        (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.3, inplace=False)
        (dropout2): Dropout(p=0.3, inplace=False)
      )
      (film): FiLMAdapter(
        (gamma): Linear(in_features=64, out_features=512, bias=True)
        (beta): Line

In [12]:
film_eval_dict = evaluate_contrastive_convergence(
        transformer_film_model,
        contrastive_model,
        val_loader,
        logits_loss_fn,
        contrastive_loss_fn,
        source_key,
        mask_token_id,
        bar_token_id,
        device
)

pprint(film_eval_dict)

validation


Epoch 0@0| val: 100%|██████████| 48/48 [00:00<00:00, 54.76batch/s, acc=0.681, floss=0.0312, hloss=0.0309, loss=0.0323]

{'val_foreign_acc': 0.57607421875,
 'val_foreign_logits_loss': 1.9769550661245983,
 'val_foreign_loss': 0.031229877145960927,
 'val_home_acc': 0.7428059895833335,
 'val_home_logits_loss': 0.7898303798089424,
 'val_home_loss': 0.030910431562612455,
 'val_no2foreign_loss': 0.031809137163994215,
 'val_no2home_acc': 0.6183430989583333,
 'val_no2home_logits_loss': 1.4203425136705239,
 'val_no2home_loss': 0.03170305696160843}


In [13]:
lacta_eval_dict = evaluate_contrastive_convergence(
        transformer_lacta_model,
        contrastive_model,
        val_loader,
        logits_loss_fn,
        contrastive_loss_fn,
        source_key,
        mask_token_id,
        bar_token_id,
        device
)

pprint(lacta_eval_dict)

validation


Epoch 0@0| val: 100%|██████████| 48/48 [00:00<00:00, 54.52batch/s, acc=0.682, floss=0.00275, hloss=0.00264, loss=0.0178]

{'val_foreign_acc': 0.581982421875,
 'val_foreign_logits_loss': 1.919294002155463,
 'val_foreign_loss': 0.002748262074116307,
 'val_home_acc': 0.7435058593749998,
 'val_home_logits_loss': 0.7891942647596201,
 'val_home_loss': 0.0026376957879013694,
 'val_no2foreign_loss': 0.03169829441079249,
 'val_no2home_acc': 0.619677734375,
 'val_no2home_logits_loss': 1.4165526169041793,
 'val_no2home_loss': 0.03150177238664279}


In [14]:
lacta_full_eval_dict = evaluate_contrastive_convergence(
        transformer_lacta_full_model,
        contrastive_model,
        val_loader,
        logits_loss_fn,
        contrastive_loss_fn,
        source_key,
        mask_token_id,
        bar_token_id,
        device
)

pprint(lacta_full_eval_dict)

validation


Epoch 0@0| val: 100%|██████████| 48/48 [00:00<00:00, 55.50batch/s, acc=0.68, floss=0.00275, hloss=0.00265, loss=0.0185] 

{'val_foreign_acc': 0.581005859375,
 'val_foreign_logits_loss': 1.9373026937246323,
 'val_foreign_loss': 0.0027471178085155166,
 'val_home_acc': 0.7424967447916666,
 'val_home_logits_loss': 0.7896366976201534,
 'val_home_loss': 0.00264564969499285,
 'val_no2foreign_loss': 0.03170569702827682,
 'val_no2home_acc': 0.616650390625,
 'val_no2home_logits_loss': 1.4243125654757023,
 'val_no2home_loss': 0.03170847919924805}


In [15]:
film_eval_dict = evaluate_contrastive_convergence(
        transformer_film_model,
        contrastive_model,
        val_loader,
        logits_loss_fn,
        contrastive_loss_fn,
        source_key,
        mask_token_id,
        bar_token_id,
        device,
        interpolate=True
)

pprint(film_eval_dict)

validation


Epoch 0@0| val: 100%|██████████| 48/48 [00:00<00:00, 55.05batch/s, acc=0.68, floss=0.0235, hloss=0.0309, loss=0.0295] 

{'val_foreign_acc': 0.6241048177083333,
 'val_foreign_logits_loss': 1.5561257985730965,
 'val_foreign_loss': 0.023539016372524202,
 'val_home_acc': 0.7434407552083332,
 'val_home_logits_loss': 0.7913074493408203,
 'val_home_loss': 0.03093294337547074,
 'val_no2foreign_loss': 0.02423776014863203,
 'val_no2home_acc': 0.6174641927083331,
 'val_no2home_logits_loss': 1.419734500348568,
 'val_no2home_loss': 0.03163646774676939}


In [16]:
lacta_eval_dict = evaluate_contrastive_convergence(
        transformer_lacta_model,
        contrastive_model,
        val_loader,
        logits_loss_fn,
        contrastive_loss_fn,
        source_key,
        mask_token_id,
        bar_token_id,
        device,
        interpolate=True
)

pprint(lacta_eval_dict)

validation


Epoch 0@0| val: 100%|██████████| 48/48 [00:00<00:00, 55.31batch/s, acc=0.679, floss=0.00437, hloss=0.00265, loss=0.017] 

{'val_foreign_acc': 0.6256184895833333,
 'val_foreign_logits_loss': 1.574888067940871,
 'val_foreign_loss': 0.004371977619787988,
 'val_home_acc': 0.742041015625,
 'val_home_logits_loss': 0.790163762246569,
 'val_home_loss': 0.0026501415801855424,
 'val_no2foreign_loss': 0.024019164227259655,
 'val_no2home_acc': 0.616015625,
 'val_no2home_logits_loss': 1.4225284829735756,
 'val_no2home_loss': 0.03157470060978085}


In [17]:
lacta_full_eval_dict = evaluate_contrastive_convergence(
        transformer_lacta_full_model,
        contrastive_model,
        val_loader,
        logits_loss_fn,
        contrastive_loss_fn,
        source_key,
        mask_token_id,
        bar_token_id,
        device,
        interpolate=True
)

pprint(lacta_full_eval_dict)

validation


Epoch 0@0| val: 100%|██████████| 48/48 [00:00<00:00, 55.37batch/s, acc=0.682, floss=0.0043, hloss=0.00265, loss=0.0167] 

{'val_foreign_acc': 0.6202799479166669,
 'val_foreign_logits_loss': 1.5834484783311684,
 'val_foreign_loss': 0.0042951255551694585,
 'val_home_acc': 0.742822265625,
 'val_home_logits_loss': 0.7880949961642424,
 'val_home_loss': 0.002647078329270395,
 'val_no2foreign_loss': 0.024106873005318146,
 'val_no2home_acc': 0.6208658854166668,
 'val_no2home_logits_loss': 1.416233063985904,
 'val_no2home_loss': 0.03155861523312827}


In [18]:
film_eval_dict = evaluate_contrastive_convergence(
        transformer_film_model,
        contrastive_model,
        val_loader,
        logits_loss_fn,
        contrastive_loss_fn,
        source_key,
        mask_token_id,
        bar_token_id,
        device,
        extrapolate=True
)

pprint(film_eval_dict)

validation


Epoch 0@0| val: 100%|██████████| 48/48 [00:00<00:00, 55.39batch/s, acc=0.681, floss=1.58, hloss=0.0309, loss=0.814]

{'val_foreign_acc': 0.02224934895833333,
 'val_foreign_logits_loss': 6.543176184097926,
 'val_foreign_loss': 1.5778350805242856,
 'val_home_acc': 0.7433756510416667,
 'val_home_logits_loss': 0.7917105332016945,
 'val_home_loss': 0.030913000848765176,
 'val_no2foreign_loss': 1.5831835220257442,
 'val_no2home_acc': 0.6184082031249999,
 'val_no2home_logits_loss': 1.4131782663365204,
 'val_no2home_loss': 0.03163815208245069}


In [19]:
lacta_eval_dict = evaluate_contrastive_convergence(
        transformer_lacta_model,
        contrastive_model,
        val_loader,
        logits_loss_fn,
        contrastive_loss_fn,
        source_key,
        mask_token_id,
        bar_token_id,
        device,
        extrapolate=True
)

pprint(lacta_eval_dict)

validation


Epoch 0@0| val: 100%|██████████| 48/48 [00:00<00:00, 55.45batch/s, acc=0.682, floss=1.32, hloss=0.00265, loss=0.745]

{'val_foreign_acc': 0.021744791666666662,
 'val_foreign_logits_loss': 6.3508180975914,
 'val_foreign_loss': 1.3205437610546749,
 'val_home_acc': 0.7448893229166668,
 'val_home_logits_loss': 0.788931871453921,
 'val_home_loss': 0.002652283670613542,
 'val_no2foreign_loss': 1.5837166955073674,
 'val_no2home_acc': 0.6188639322916667,
 'val_no2home_logits_loss': 1.4235481098294258,
 'val_no2home_loss': 0.0316120299588268}


### Linearity and curvature of the contrastive space

Interpolation can occur either before or after the contrastive space. To do that, we examine how much the interpolated values of random pairs before the contrastive space fall close to the respective interpolated values after the contrastive space.

In [61]:
diff = 0

for batch in val_loader:
    # mix the batch twice
    mixed_batch_1 = make_mixed_batch(batch, source_key)
    mixed_batch_2 = make_mixed_batch(mixed_batch_1, source_key)
    # get interpolated values
    before_contrastive = (mixed_batch_1[source_key].to(device) + mixed_batch_2[source_key].to(device))/2
    # get contrastive image
    pre_contrastive_image = contrastive_model.source_proj(before_contrastive.to(device))
    # get each image and mix after
    contra_image_1 = contrastive_model.source_proj(mixed_batch_1[source_key].to(device))
    contra_image_2 = contrastive_model.source_proj(mixed_batch_2[source_key].to(device))
    # mix after
    post_contrastive_mix = (contra_image_1 + contra_image_2)/2
    
    diff += torch.norm((pre_contrastive_image - post_contrastive_mix))/(torch.norm(contra_image_2)*len(batch))
diff /= len(val_loader)
print(diff)

tensor(0.0846, device='cuda:0', grad_fn=<DivBackward0>)


### Going in-between places

Mix two batches and try to go "in the middle" of two guiding vectors no the respective poisitions of the mixed batch.

In [49]:
batch = next(iter(val_loader))

In [51]:
melody_grid = batch["pianoroll"].to(device)
harmony_gt = batch["harmony_ids"].to(device)
home_guidance_embeddings = batch[source_key].to(device)
# mix the batch twice
mixed_batch_1 = make_mixed_batch(batch, source_key)
mixed_batch_2 = make_mixed_batch(mixed_batch_1, source_key)

# foreign_guidance_embeddings = 5000*mixed_batch_1[source_key].to(device) + 0.5*mixed_batch_2[source_key].to(device)

z_guidance_1 = contrastive_model.source_proj(mixed_batch_1[source_key].to(device))
z_guidance_2 = contrastive_model.source_proj(mixed_batch_2[source_key].to(device))

In [43]:
tmp_embs = []
for m in [1, 100, 10000]:
    foreign_guidance_embeddings = m*mixed_batch_1[source_key].to(device)
    z_guidance = contrastive_model.source_proj(foreign_guidance_embeddings.to(device))
    print(z_guidance[0])
    tmp_embs.append(z_guidance[0])

tensor([-4.0743e-02,  5.6031e-03,  1.9911e-01, -2.2012e-01, -4.9935e-02,
        -2.3820e-01,  3.4029e-02, -8.4535e-02, -1.0419e-01,  1.3795e-01,
        -6.8604e-02, -7.2682e-02,  4.5733e-02,  1.2904e-01,  1.5622e-01,
         1.4202e-01, -3.4287e-02,  2.1385e-01, -4.0956e-02,  2.5835e-02,
        -2.8173e-01, -4.9625e-02,  8.3936e-02, -1.2192e-01,  1.6789e-01,
         7.2504e-02,  7.6067e-02,  7.6011e-02, -8.6672e-02, -7.6127e-02,
         2.3178e-01,  1.5733e-01,  4.2478e-02, -2.4770e-01, -4.8193e-02,
        -5.0679e-03,  1.0134e-01,  1.9619e-01, -1.3093e-01,  9.5799e-02,
         8.2007e-02,  2.3181e-01, -1.6890e-01, -3.5116e-02,  1.1046e-02,
        -4.6970e-02,  2.4006e-01,  2.4027e-04, -1.3755e-01, -5.9233e-02,
         2.2113e-04,  1.0101e-01, -1.6510e-01, -9.4364e-02,  1.2648e-01,
        -7.7990e-02, -1.9297e-01,  2.5400e-02, -7.8463e-02,  1.3192e-02,
         9.0016e-02, -4.9913e-02,  1.0925e-01,  3.6169e-02], device='cuda:0',
       grad_fn=<SelectBackward0>)
tensor([-2.2

In [44]:
num_visible = 0

harmony_input, harmony_target = full_to_partial_masking(
    harmony_gt,
    mask_token_id,
    num_visible,
    bar_token_id=bar_token_id
)

# Step 1: contrastive latent attraction validation
z_guidance = contrastive_model.source_proj(foreign_guidance_embeddings.to(device))
logits, hidden = transformer_model(
    melody_grid.to(device),
    harmony_input.to(device),
    z_guidance.to(device),
    return_hidden=True
)
z_guidance, z_transformer, _ = contrastive_model( foreign_guidance_embeddings.to(device), hidden.to(device))
foreign_guidance_loss = contrastive_loss_fn(z_guidance,z_transformer)

# Step 2: home attraction validation
z_guidance = contrastive_model.source_proj(home_guidance_embeddings.to(device))
logits, hidden = transformer_model(
    melody_grid.to(device),
    harmony_input.to(device),
    z_guidance.to(device),
    return_hidden=True
)
z_guidance, z_transformer, _ = contrastive_model( home_guidance_embeddings.to(device), hidden.to(device))
home_guidance_loss = contrastive_loss_fn(z_guidance,z_transformer)

home_logits_loss = logits_loss_fn(logits.view(-1, logits.size(-1)), harmony_target.view(-1))

# Step 3: no attraction
logits, hidden = transformer_model(
    melody_grid.to(device),
    harmony_input.to(device),
    None,
    return_hidden=True
)
z_guidance, z_transformer, _ = contrastive_model( home_guidance_embeddings.to(device), hidden.to(device))
no_guidance_to_home_loss = contrastive_loss_fn(z_guidance,z_transformer)

no_guidance_to_home_logits_loss = logits_loss_fn(logits.view(-1, logits.size(-1)), harmony_target.view(-1))

z_guidance, z_transformer, _ = contrastive_model( foreign_guidance_embeddings.to(device), hidden.to(device))
no_guidance_to_foreign_loss = contrastive_loss_fn(z_guidance,z_transformer)

In [45]:
print('foreign_guidance_loss: ', foreign_guidance_loss.item())
print('home_guidance_loss: ', home_guidance_loss.item())
print('no_guidance_to_home_loss: ', no_guidance_to_home_loss.item())
print('no_guidance_to_foreign_loss: ', no_guidance_to_foreign_loss.item())
print(10*'-', 'logits', 10*'-')
print('home_logits_loss: ', home_logits_loss.item())
print('no_guidance_to_home_logits_loss: ', no_guidance_to_home_logits_loss.item())

foreign_guidance_loss:  0.0031875669956207275
home_guidance_loss:  0.0025094535667449236
no_guidance_to_home_loss:  0.031480573117733
no_guidance_to_foreign_loss:  0.03120112046599388
---------- logits ----------
home_logits_loss:  0.5404249429702759
no_guidance_to_home_logits_loss:  1.245413064956665


In [ ]:
tqdm_position = 0
epoch = 0
step = 0
num_visible = 0

with torch.no_grad():
    running_foreign_loss = 0
    val_foreign_loss = 0

    running_home_loss = 0
    val_home_loss = 0

    running_no2home_loss = 0
    val_no2home_loss = 0

    running_no2foreign_loss = 0
    val_no2foreign_loss = 0

    # logits losses
    running_home_logits_loss = 0
    val_home_logits_loss = 0
    running_no2home_logits_loss = 0
    val_no2home_logits_loss = 0

    # accuracies
    running_home_acc = 0
    val_home_acc = 0
    running_no2home_acc = 0
    val_no2home_acc = 0

    batch_num = 0
    print('validation')
    with tqdm(val_loader, unit='batch', position=tqdm_position) as tepoch:
        tepoch.set_description(f'Epoch {epoch}@{step}| val')
        for batch in tepoch:
            batch_num += 1
            melody_grid = batch["pianoroll"].to(device)
            harmony_gt = batch["harmony_ids"].to(device)
            home_guidance_embeddings = batch[source_key].to(device)
            mixed_batch_1 = make_mixed_batch(batch, source_key)
            mixed_batch_2 = make_mixed_batch(mixed_batch_1, source_key)
            foreign_guidance_embeddings_1 = mixed_batch_1[source_key].to(device)
            foreign_guidance_embeddings_2 = mixed_batch_2[source_key].to(device)

            harmony_input, harmony_target = full_to_partial_masking(
                harmony_gt,
                mask_token_id,
                num_visible,
                bar_token_id=bar_token_id
            )

            # Step 1: contrastive latent attraction validation
            z_guidance_1 = contrastive_model.source_proj(foreign_guidance_embeddings_1.to(device))
            z_guidance_2 = contrastive_model.source_proj(foreign_guidance_embeddings_2.to(device))
            z_guidance = 0.5*z_guidance_1 + 0.5*z_guidance_2
            logits, hidden = transformer_model(
                melody_grid.to(device),
                harmony_input.to(device),
                z_guidance.to(device),
                return_hidden=True
            )
            z_guidance, z_transformer, _ = contrastive_model( foreign_guidance_embeddings.to(device), hidden.to(device))
            foreign_guidance_loss = contrastive_loss_fn(z_guidance,z_transformer)

            # Step 2: home attraction validation
            z_guidance = contrastive_model.source_proj(home_guidance_embeddings.to(device))
            logits, hidden = transformer_model(
                melody_grid.to(device),
                harmony_input.to(device),
                z_guidance.to(device),
                return_hidden=True
            )
            z_guidance, z_transformer, _ = contrastive_model( home_guidance_embeddings.to(device), hidden.to(device))
            home_guidance_loss = contrastive_loss_fn(z_guidance,z_transformer)

            running_home_logits_loss += logits_loss_fn(logits.view(-1, logits.size(-1)), harmony_target.view(-1)).item()
            val_home_logits_loss = running_home_logits_loss/batch_num

            # accuracy
            predictions = logits.argmax(dim=-1)
            # mask = torch.logical_and(harmony_target != harmony_input, harmony_target != -100)
            mask = harmony_target != -100
            running_home_acc += (predictions[mask] == harmony_target[mask]).sum().item()/mask.sum().item()
            val_home_acc = running_home_acc/batch_num

            # partial losses
            running_foreign_loss += foreign_guidance_loss.item()
            val_foreign_loss = running_foreign_loss/batch_num
            running_home_loss += home_guidance_loss.item()
            val_home_loss = running_home_loss/batch_num

            # Step 3: no attraction
            logits, hidden = transformer_model(
                melody_grid.to(device),
                harmony_input.to(device),
                None,
                return_hidden=True
            )
            z_guidance, z_transformer, _ = contrastive_model( home_guidance_embeddings.to(device), hidden.to(device))
            no_guidance_to_home_loss = contrastive_loss_fn(z_guidance,z_transformer)

            no_guidance_to_home_logits_loss = logits_loss_fn(logits.view(-1, logits.size(-1)), harmony_target.view(-1))

            z_guidance, z_transformer, _ = contrastive_model( foreign_guidance_embeddings.to(device), hidden.to(device))
            no_guidance_to_foreign_loss = contrastive_loss_fn(z_guidance,z_transformer)

            running_no2home_logits_loss += logits_loss_fn(logits.view(-1, logits.size(-1)), harmony_target.view(-1)).item()
            val_no2home_logits_loss = running_no2home_logits_loss/batch_num

            # accuracy
            predictions = logits.argmax(dim=-1)
            # mask = torch.logical_and(harmony_target != harmony_input, harmony_target != -100)
            mask = harmony_target != -100
            running_no2home_acc += (predictions[mask] == harmony_target[mask]).sum().item()/mask.sum().item()
            val_no2home_acc = running_no2home_acc/batch_num

            # partial losses
            running_no2home_loss += no_guidance_to_home_loss.item()
            val_no2home_loss = running_no2home_loss/batch_num
            running_no2foreign_loss += no_guidance_to_foreign_loss.item()
            val_no2foreign_loss = running_no2foreign_loss/batch_num

            loss = 0.25*home_guidance_loss + 0.25*foreign_guidance_loss + 0.25*no_guidance_to_home_loss + 0.25*no_guidance_to_foreign_loss
            acc = 0.5*val_home_acc + 0.5*val_no2home_acc

            tepoch.set_postfix(
                loss=loss.item(),
                floss=val_foreign_loss,
                hloss=val_home_loss,
                acc=acc
            )
        # end for batch
    # end with tqdm
# end with no grad

validation


Epoch 0@0| val: 100%|██████████| 48/48 [00:00<00:00, 52.51batch/s, acc=0.682, floss=0.00269, hloss=0.00266, loss=0.0198]


In [47]:
print('val_foreign_loss: ', val_foreign_loss)
print('val_home_loss: ', val_home_loss)
print('val_no2home_loss: ', val_no2home_loss)
print('val_no2foreign_loss: ', val_no2foreign_loss)

print(10*'-', 'logits', 10*'-')
print('val_home_logits_loss: ', val_home_logits_loss)
print('val_no2home_logits_loss: ', val_no2home_logits_loss)

print(10*'-', 'accuracy', 10*'-')
print('val_home_acc: ', val_home_acc)
print('val_no2home_acc: ', val_no2home_acc)

val_foreign_loss:  0.002687020076943251
val_home_loss:  0.0026579652233825377
val_no2home_loss:  0.03163425647653639
val_no2foreign_loss:  0.034400557012607656
---------- logits ----------
val_home_logits_loss:  0.7926508268962303
val_no2home_logits_loss:  1.4174690345923107
---------- accuracy ----------
val_home_acc:  0.7431152343749999
val_no2home_acc:  0.6207356770833333
